In [2]:
import time
from itertools import combinations
# Dataset:
dataset = [
    ['leche', 'pan', 'mantequilla', 'huevos', 'queso'],
    ['pan', 'mantequilla', 'mermelada', 'jugo'],
    ['leche', 'pan', 'huevos', 'cereal'],
    ['leche', 'mermelada', 'jugo'],
    ['pan', 'mantequilla', 'queso'],
    ['leche', 'pan', 'mantequilla', 'mermelada', 'cereal'],
    ['pan', 'huevos', 'jugo'],
    ['mantequilla', 'mermelada', 'queso'],
    ['leche', 'huevos', 'cereal'],
    ['pan', 'mantequilla', 'huevos', 'queso'],
    ['pan', 'jugo'],
    ['leche', 'pan', 'mantequilla', 'queso'],
    ['mermelada', 'cereal'],
    ['pan', 'mantequilla', 'jugo'],
    ['leche', 'queso', 'huevos']
]



## Apriori propio

In [33]:
from itertools import combinations
import time

# Set the minimum support threshold
threshold = 0.25

# Count itemsets of any length k
def count_itemsets(dataset, candidates):
    itemset_count = {}
    for transaction in dataset:
        transaction = set(transaction)
        for candidate in candidates:
            if set(candidate).issubset(transaction):
                candidate = tuple(sorted(candidate))
                itemset_count[candidate] = itemset_count.get(candidate, 0) + 1
    return itemset_count

# Calculate support
def calculate_support(itemset_count, total_transactions):
    return {itemset: count / total_transactions for itemset, count in itemset_count.items()}

# Generate candidate itemsets of size k from frequent (k-1)-itemsets
def generate_candidates(prev_frequent_itemsets, k):
    items = set()
    for itemset in prev_frequent_itemsets:
        items.update(itemset)
    return list(combinations(sorted(items), k))

# Filter itemsets by support threshold
def filter_itemsets_by_support(supports, threshold):
    return {itemset: support for itemset, support in supports.items() if support >= threshold}

# Apriori algorithm
def apriori(dataset, threshold):
    total_transactions = len(dataset)
    frequent_itemsets = {}
    
    # Step 1: Find frequent 1-itemsets
    item_count = count_itemsets(dataset, [(item,) for transaction in dataset for item in transaction])
    support = calculate_support(item_count, total_transactions)
    current_frequent = filter_itemsets_by_support(support, threshold)
    k = 2
    frequent_itemsets.update(current_frequent)

    # Step 2: Generate larger itemsets
    while current_frequent:
        candidates = generate_candidates(current_frequent.keys(), k)
        itemset_count = count_itemsets(dataset, candidates)
        support = calculate_support(itemset_count, total_transactions)
        current_frequent = filter_itemsets_by_support(support, threshold)
        frequent_itemsets.update(current_frequent)
        k += 1
    
    return frequent_itemsets

# Generate association rules
def generate_rules(frequent_itemsets, min_confidence):
    rules = []
    for itemset in frequent_itemsets:
        if len(itemset) < 2:
            continue
        itemset_support = frequent_itemsets[itemset]
        for i in range(1, len(itemset)):
            for antecedent in combinations(itemset, i):
                antecedent = tuple(sorted(antecedent))
                consequent = tuple(sorted(set(itemset) - set(antecedent)))
                if consequent and antecedent in frequent_itemsets:
                    confidence = itemset_support / frequent_itemsets[antecedent]
                    if confidence >= min_confidence:
                        rules.append((antecedent, consequent, itemset_support, confidence))
    return rules


## Apriori Avanzado

In [34]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori as mlxtend_apriori
from mlxtend.frequent_patterns import association_rules as mlxtend_rules

# Step 1: Encode the dataset
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)

## Comparación

In [31]:
# Run ustom Apriori
start = time.time()
frequent_itemsets = apriori(dataset, threshold)
rules = generate_rules(frequent_itemsets, 0.5)
end = time.time()

print(f"Execution time: {end - start:.4f} seconds")
print(f"Frequent itemsets found: {len(frequent_itemsets)}")
print(f"Rules generated: {len(rules)}")
print("Example rules:")
for antecedent, consequent, supp, conf in rules[:5]:
    print(f"{antecedent} => {consequent} (supp: {supp:.2f}, conf: {conf:.2f})")

# Run mlxtend Apriori
start_mlxtend = time.time()
frequent_itemsets_mlxtend = mlxtend_apriori(df, min_support=0.25, use_colnames=True)
rules_mlxtend = mlxtend_rules(frequent_itemsets_mlxtend, metric="confidence", min_threshold=0.5)
end_mlxtend = time.time()

# Output results
print(f"\n--- mlxtend Apriori ---")
print(f"Execution time: {end_mlxtend - start_mlxtend:.4f} seconds")
print(f"Frequent itemsets found: {frequent_itemsets_mlxtend.shape[0]}")
print(f"Rules generated: {rules_mlxtend.shape[0]}")
print("Example rules:")
for i, row in rules_mlxtend.head(5).iterrows():
    antecedent = ', '.join(list(row['antecedents']))
    consequent = ', '.join(list(row['consequents']))
    print(f"{antecedent} => {consequent} (supp: {row['support']:.2f}, conf: {row['confidence']:.2f})")

Execution time: 0.0017 seconds
Frequent itemsets found: 16
Rules generated: 3
Example rules:
('mantequilla', 'pan') => ('queso',) (supp: 0.27, conf: 0.57)
('mantequilla', 'queso') => ('pan',) (supp: 0.27, conf: 0.80)
('pan', 'queso') => ('mantequilla',) (supp: 0.27, conf: 1.00)

--- mlxtend Apriori ---
Execution time: 0.0144 seconds
Frequent itemsets found: 16
Rules generated: 15
Example rules:
huevos => leche (supp: 0.27, conf: 0.67)
leche => huevos (supp: 0.27, conf: 0.57)
huevos => pan (supp: 0.27, conf: 0.67)
jugo => pan (supp: 0.27, conf: 0.80)
leche => pan (supp: 0.27, conf: 0.57)


El apriori custom corre más rápido pero no detecta las reglas con items iniciales, a diferencia del mlxtend